# 19 · 第六个工具：看基因检测结果（表型 + 基因）

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → 17 找相似病人 ✅ → 18 疾病介绍（RAG） ✅ → [19 基因工具 ✅] → 20 大考
```

> **这一章的代码在合并请求 #21 里。** 不用模型的评测免费；调模型的只有两次 20 道题的试跑（合计约 ¥1），记在 MLflow 里。下面每一格都只读记录、在本机计算，**运行这本笔记不花钱**。

## 背景：医院里确诊罕见病，靠的是基因检测

一个疑似罕见病的孩子来看病，现在的常规做法是做基因检测（外显子组测序）。测完、按标准过滤以后，会剩下几十个"可疑基因"，其中最多一个是真正致病的。医生要回答的问题就变成了：**这几十个基因里，哪个能解释这个孩子的症状？**

医院常用的软件 Exomiser 做的就是这件事。我们之前的工具都只看症状，这一章加上基因。

**先更正我之前的一个说法。** 第 16 章我说基因工具能补上 ① 类那 40 道题（病只在这道题自己的论文里描述过），这是错的。"这个基因导致这个病"本身就是这篇论文首次报告的，按防泄露的规则 3（见 [防泄露说明](../leakage.md)），这条知识也得遮住。基因工具补不上这 40 道，它的用处是让其余的题容易得多。

## 这一章做的 6 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 新考法：给每道题模拟一份基因检测结果 | 真基因 + 49 个假基因，固定随机种子 | 我们的数据只记了真基因，没有完整的测序结果 |
| 2 | 新工具：按症状给候选基因排名 | 每个基因已知的病 × 症状相似度 | 几十个基因里找出致病的那个 |
| 3 | 防泄露 | 规则 3 也管"基因–疾病"这条知识 | 只从本篇论文才知道的关联不能用 |
| 4 | 不用模型先考 | 新基线 `gene-rank` | 基因本身值多少 |
| 5 | 接进 agent：试跑、看错题、改一句话 | ci-20 两次试跑 | agent 不把"没测出来"当证据 |
| 6 | 两种考法分开报告 | 独立的实验和数据集 | 不拿容易的考试和难的考试比 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 新考法：模拟一份基因检测结果

### 在干什么
给每道题配一份"基因检测结果"：50 个候选基因，其中 1 个是真正致病的，49 个是假的。

### 背景
Phenopacket Store 只记了每个病人的致病基因，没有完整的测序结果。只把真基因告诉 agent，等于直接给答案；所以要造一份像真的一样的候选名单，把真基因藏在里面。

### 用了什么技术，为什么
| 规则 | 为什么 |
|---|---|
| 1 个真基因 + 49 个假基因 | 测序过滤后、只看已知致病基因，一般剩几十个，取 50 |
| 假基因从 HPO 里已知会致病的基因里抽 | 全是"看起来都有可能"的基因，题目不至于太容易 |
| 和这道题的病有关的基因不抽 | 免得出现"假基因其实也对" |
| 用题目编号做随机种子 | 每次考试都是同一份名单，结果能复现 |
| 按字母排序 | 真基因的位置不透露任何信息 |

**简化的地方**（每次报告结果都要写明）：只给基因名，不给具体的变异、杂合还是纯合、致病性评分；假基因是均匀随机抽的，而真实测序里更容易出现很长、很容易变异的基因。

### 公司里什么情况怎么做
- **数据不够就模拟，但要把模拟的方式写清楚**：别人才能判断结果能不能信
- **模拟要固定种子**：每次结果一样，才能比较

### 执行
看一道真实的题：病人的症状，和模拟出来的 50 个候选基因。

In [ ]:
from raredx.config import settings
from raredx.data import cases as case_store, sources
from raredx.evals.candidates import SPEC, candidate_genes
from raredx.knowledge import hpo as hpo_knowledge, masking

registry = sources.load_sources(ROOT / "evals/datasets.toml")
hpo = hpo_knowledge.load(sources.raw_dir(registry["hpo"], ROOT / "data"))
cases = {c.id: c for c in case_store.read(sources.processed_dir(registry["phenopacket-store"], ROOT / "data"))}

case = cases["PMID_30509212_male"]
genes = candidate_genes(case, hpo)
print("规则：", SPEC, "\n")
print("症状：", "、".join(case.hpo_labels), "\n")
print(f"{len(genes)} 个候选基因：", ", ".join(genes))

### 结果怎么看
- 这个病人有长睑裂、下眼睑外翻、腭裂、生长迟缓……这是歌舞伎综合征的典型面容
- 50 个基因按字母排着，看不出哪个是真的。答案在下一步揭晓

### 面试怎么说
> "Phenopacket Store records the causal gene but not the exome, so the genotype track simulates one: the causal gene among 49 decoys drawn per case, with a fixed seed, from known disease genes not linked to the case's disease, sorted by symbol. The simplifications are stated with every result."

---

## 第 2 步 · 新工具：按症状给候选基因排名

### 在干什么
新工具 `rank_candidate_genes`：给它症状和候选基因，它查出每个基因已知会导致哪些病，算这些病和病人的症状有多像，把基因排好序。

### 背景
一个基因可能导致好几种病。只要其中一种病和病人很像，这个基因就很可疑。

### 用了什么技术，为什么
- **"基因–疾病"关系**：HPO 自带的表（来自 OMIM），只用"孟德尔遗传"那一类
- **相似度和排名工具一模一样**（Resnik，funSimAvg）：每个基因取它最像的那个病的分数
- **没有已知疾病的基因单独列出来**：它也可能是致病的（一个新病），只是没法打分，不能直接扔掉
- 这是 Exomiser 里"看表型"那一半的简化版；"看变异"那一半（人群频率、预测的危害程度）没有做

### 公司里什么情况怎么做
- **结论要能追溯**：每个基因都附上它的病和分数，医生能看懂为什么排在前面
- **不确定的东西单独放**：没法打分的基因列在一边，不混进排名

### 执行

In [ ]:
import mlflow
from raredx.tools.rank_candidate_genes import RankCandidateGenes

mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment("playground")
rank_genes = RankCandidateGenes(hpo)
with masking.masked([case.pmid]):                 # 和考试时一样，遮住这道题自己的论文
    ranking = rank_genes(case.hpo_ids, genes, top_k=5)
for m in ranking.genes:
    best = m.diseases[0]
    print(f"{m.gene:<8} 分数 {m.best_score:.2f}   最像的病：{best.disease_id} {best.disease_name}")
print("\n没有已知疾病的候选基因：", ranking.no_known_disease or "无")
print("\n答案：", case.gene, "→", case.disease_id, case.disease_label)

### 结果怎么看
- 排第 1 的是 KDM6A，它导致歌舞伎综合征 2 型，就是答案
- 后面几个基因的病分数低很多：它们导致的病和这个孩子不像

### 面试怎么说
> "The gene tool scores each candidate by the best phenotype match among the diseases it is known to cause — the phenotype half of what Exomiser does. Genes with no known disease are listed apart: possibly causal, but not scorable."

---

## 第 3 步 · 防泄露

### 在干什么
确认规则 3 对基因也生效：一个病只从这道题的论文才知道，那么"这个基因导致这个病"也不能用。

### 背景
dev-200 里有 40 道题，病只在这道题自己的论文里描述过，往往这篇论文就是第一次发现"这个基因导致这个病"。真实世界里，医生在这种时候拿到的是一个"还没有已知疾病"的基因。

### 用了什么技术，为什么
工具在查每个基因的病时，去掉"只从被遮论文才知道的病"。如果一个基因的病全被去掉，它就进了"没有已知疾病"那一栏，和真实情况一样。

### 执行
找一道这样的题，看它的真基因在工具里是什么样子，再数一数 40 道题里各是什么情况。

In [ ]:
from raredx.evals import splits

dev = [cases[i] for i, _ in splits.read(ROOT / "evals/splits/dev-200.tsv")]
mlflow.tracing.disable()          # 本机计算，不记追踪
unscored = scored_elsewhere = 0
example = None
for c in dev:
    if c.disease_id not in hpo.known_only_from(frozenset({c.pmid})):
        continue
    with masking.masked([c.pmid]):
        result = rank_genes(c.hpo_ids, candidate_genes(c, hpo), top_k=50)
    if c.gene in result.no_known_disease:
        unscored += 1
        example = example or (c, result)
    else:
        scored_elsewhere += 1
mlflow.tracing.enable()
c, result = example
print("例子：", c.disease_id, c.disease_label, " 真基因", c.gene, " 出处", c.pmid)
print("  工具里，", c.gene, "在不在'没有已知疾病'那一栏：", c.gene in result.no_known_disease)
print(f"\n40 道题里：真基因显示为'没有已知疾病' {unscored} 道；真基因还关联着别的病、照样被打分 {scored_elsewhere} 道")

### 结果怎么看
- 例子里的真基因，在工具眼里是"没有已知疾病"：和医生第一次遇到这种病人时一样
- 另外那些题，真基因还关联着别的（有别的论文记载的）病，所以照样被打分；但正确答案那个病对所有工具都不存在，不算泄露
- 所以这种考法的**上限是 80%**：40 道题的病，工具无论如何都查不到

### 面试怎么说
> "The masking rule covers gene–disease links too: when a disease is known only from the case's own paper, that paper is usually where the gene was linked to it, so the gene shows as one with no known disease — as it would to a clinician seeing the first such patient. That caps this track at 80% here."

---

## 第 4 步 · 不用模型，先考工具自己

### 在干什么
不用模型的基线 `gene-rank`：按工具给的基因排名，把这些基因的病依次列出来，在 dev-200 上考。

### 执行

In [ ]:
from raredx.evals.harness import Harness

harness = Harness(settings.mlflow_tracking_uri, settings.mlflow_artifact_root)
def latest(name, split):
    return next(r for r in harness.runs(split) if r["name"] == name and not r["code"].endswith("-dirty"))
for label, name, split in (("只看症状：按症状排名", "phenotype-rank", "dev-200"),
                           ("症状 + 50 个候选基因：按基因排名", "gene-rank", "dev-200+genes")):
    m = harness.client.get_run(latest(name, split)["run_id"]).data.metrics
    print(f"{label:<20} 第 1 名 {m['disease_top1/mean']:.1%}   前 10 名 {m['disease_top10/mean']:.1%}   "
          f"找对基因（第 1 名）{m['gene_top1/mean']:.1%}")

### 结果怎么看

| 不用模型 | 第 1 名 | 前 10 名 |
|---|---|---|
| 只看症状 | 19.0% | 49.0% |
| **症状 + 50 个候选基因** | **62.0%** | **78.5%** |

- **多了基因，题目一下容易了很多**：这正是医院里基因检测是确诊主力的原因
- 前 10 名 78.5%，离上限 80% 只差一点
- **这两行不能拿来说"我们的系统进步了"**：下面那行是另一种考试，多给了一份信息

### 面试怎么说
> "With fifty candidate genes, ranking them by phenotype alone gets 62% top-1 against 19% for phenotype-only ranking — but that is a different exam with more information, so it is reported apart, never as a gain."

---

## 第 5 步 · 接进 agent：试跑、看错题、改一句话

### 在干什么
题目里有候选基因时，agent 能看到这份名单，也多了基因工具。先用 ci-20 试跑。

### 背景
第一次试跑，agent 的第 1 名只有 45%，**比不用模型的基因排名（60%）还低**，虽然 20 道题它每道都用了基因工具。翻错题，原因很清楚：**它选了基因根本不在候选名单里的病。**

### 执行
从第一次试跑的记录里，调出一道这样的题。

In [ ]:
from raredx.evals.harness import candidates_of

runs = [r for r in harness.runs("ci-20+genes") if r["name"] == "seeded-agent"]
after, before = runs[0], runs[-1]                  # 最新一次在前
trace = next(t for t in harness.traces(before["run_id"])
             if t.data.spans[0].inputs["question"]["case_id"] == "PMID_30509212_male")
first = candidates_of(trace.data.spans[0].outputs)[0]
print("第一次试跑，agent 的第 1 名：", first.disease_name, "  基因", first.gene)
print("它的理由：", first.reason)

### 结果怎么看
- 就是第 1 步那个歌舞伎综合征的孩子。测序查出的是 KDM6A（2 型的基因），agent 却把 1 型排第 1，而 1 型的基因 KMT2D 根本不在名单里
- 另一道题，它的理由里自己写着"这个基因不在测序候选里"，可还是排了第 1
- 它只看症状像不像，没把"做过基因检测、这个基因没查出来"当成反对的证据。医生不会这么判断

**改法**：在题目里列候选基因的那一行后面，加一句"病人做过基因检测：一个病的基因如果都不在这份名单里，不管症状多像，都不太可能"。只看症状的考试不受影响，提示词的其余部分一字不变。

### 执行

In [ ]:
rows = [("不用模型：按基因排名", latest("gene-rank", "ci-20+genes")),
        ("agent，第一次试跑", before), ("agent，加了那句话以后", after)]
for label, record in rows:
    m = harness.client.get_run(record["run_id"]).data.metrics
    print(f"{label:<14} 第 1 名 {m['disease_top1/mean']:.0%}   前 10 名 {m['disease_top10/mean']:.0%}   "
          f"泄露审计 {harness.client.get_run(record['run_id']).data.tags.get('leak_audit')}")

### 结果怎么看
- 加了一句话，第 1 名 45% → 60%，追平了不用模型的基因排名
- 20 道题太少，误差范围有正负 20 个百分点，**不能说 agent 比基因排名强**。结论留到第 20 章的 200 道题
- 两次试跑都通过了泄露审计

### 公司里什么情况怎么做
- **小规模试跑不只看"能不能跑通"，也要翻错题**：这次就是从 4 道错题里看出了一个系统性的问题
- **改提示词要有针对性**：一句话说清楚一条推理规则，比整段重写更好，也不影响别的考试

### 面试怎么说
> "The first smoke run of the genotype track had the agent below the no-model gene ranking: it ranked first diseases whose genes sequencing had not found — once saying so in its own reason. One sentence on what sequencing implies took it from 45% to 60% top-1 on those 20 cases; the 200-case run will say whether that holds."

---

## 第 6 步 · 两种考法分开报告

### 在干什么
"只看症状"和"症状 + 基因"是两种考试，分开记录，分开报告。

### 用了什么技术，为什么
- `raredx eval run --with-genes` 跑的是基因考法，记在单独的实验里（`eval/dev-200+genes`），题目也登记成单独的数据集
- 候选名单的规则有版本号（现在是 v3），规则一改，旧的题目就不能再用（这一章里改过两次：去掉了没有症状记录的病，去掉了没有名字的"基因"）

### 公司里什么情况怎么做
- **不同条件下的分数不能放在一起比**：多给了信息的考试分数高，不代表系统进步了
- **测试题的生成规则也要有版本**：规则变了，题目就变了，旧的结果不能和新的混在一起

### 面试怎么说
> "Phenotype-only and phenotype-plus-genes are separate exams with separate experiments and datasets; the candidate-list rules are versioned, so a change to them can never silently mix old questions with new."

---

## 本章小结

| 做了什么 | 结果 |
|---|---|
| 新考法 | 每道题 50 个候选基因（1 真 49 假），固定种子、按字母排序 |
| 新工具 `rank_candidate_genes` | 按每个基因的病和症状像不像，给基因排名 |
| 防泄露 | 只从本篇论文才知道的"基因–疾病"关系也遮住；上限 80% |
| 不用模型先考 | 第 1 名 62.0%，前 10 名 78.5%（只看症状是 19.0%、49.0%） |
| 接进 agent | 试跑发现它忽略"没测出来"；加一句话后 ci-20 第 1 名 45% → 60% |

## 面试题

1. **How do you evaluate a genotype tool without exome data?**
   > "Simulate the exome conservatively and say so: the causal gene among 49 seeded decoys from known disease genes, none linked to the case's disease, sorted. Report it as its own exam, with its simplifications."

2. **Why can't the gene tool solve the cases whose disease was first described in their own paper?**
   > "Because the gene–disease link comes from that same paper. Masking it is what a clinician faces with the first such patient: a candidate gene with no known disease. That caps the track at 80% here, and that is the honest number."

3. **What did the smoke run teach you?**
   > "To read the errors, not only the pass rate: the agent was discounting the sequencing evidence. One sentence fixed a systematic reasoning gap."

**下一章 20 · 大考**：工具全部做完了。在 dev-200 上，带全部工具考一次，再每次拿掉一个工具各考一次，看每个工具值多少分；两种考法分开考。